<a href="https://colab.research.google.com/github/CalculatedContent/xgbwwdata/blob/main/XGBWW_OpenML_1049_W7W8_Alpha_Checkpoint.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# OpenML 1049: W7/W8 alpha trajectory with checkpoint + resume

This notebook trains one XGBoost model on **OpenML dataset 1049** and computes WeightWatcher `alpha` for **W7** and **W8** at every evaluation round.
It checkpoints to Google Drive so you can safely restart the runtime and resume where you left off.


## 1) Mount Google Drive + configure run


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

RUN_NAME = "openml_1049_w7w8_alpha"
DRIVE_ROOT = Path('/content/drive/MyDrive/xgbww_runs') / RUN_NAME
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EVERY_ROUNDS = 1   # evaluate/save every N rounds
MAX_ROUNDS = 1000             # safety cap
TARGET_ALPHA = 2.0            # stop when alpha(W7) and alpha(W8) <= this value
TEST_SIZE = 0.2
RANDOM_STATE = 42
FORCE_FRESH_START = False     # True = ignore prior checkpoints and start over
RESTART_RUNTIME_AFTER_INSTALL = False

print('Checkpoint folder:', DRIVE_ROOT)
print('Started at:', datetime.utcnow().isoformat() + 'Z')


## 2) Fresh install from source (with optional packages)


In [ ]:
import os, sys, subprocess

# Fresh clone each run (as requested)
!rm -rf /content/repo_xgbwwdata
!git clone https://github.com/CalculatedContent/xgbwwdata.git /content/repo_xgbwwdata

# Core libs
core_pkgs = [
    'xgboost', 'weightwatcher', 'torch', 'openml', 'pmlb', 'scikit-learn', 'pandas', 'numpy', 'scipy'
]
for pkg in core_pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Optional package: keel-ds (skip if unavailable)
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'keel-ds'])
    print('Installed optional package: keel-ds')
except Exception as e:
    print('Optional package keel-ds unavailable; continuing without it.')
    print(type(e).__name__, e)

# Install xgbwwdata from source repo
%run /content/repo_xgbwwdata/scripts/colab_install.py --repo /content/repo_xgbwwdata

if RESTART_RUNTIME_AFTER_INSTALL:
    print('Restart requested. Re-run notebook after runtime restart to continue.')
    os.kill(os.getpid(), 9)


## 3) Imports + helpers


In [ ]:
import gc
import time
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import torch
import weightwatcher as ww

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

from xgbwwdata import Filters, load_dataset
from xgboost2ww import convert

STATE_PATH = DRIVE_ROOT / 'state.json'
METRICS_PATH = DRIVE_ROOT / 'metrics.csv'
MODEL_PATH = DRIVE_ROOT / 'model_latest.json'
SPLIT_PATH = DRIVE_ROOT / 'data_split.npz'
SUMMARY_PATH = DRIVE_ROOT / 'summary.json'


def _extract_weight_shape(layer):
    candidates = []

    if hasattr(layer, 'weight'):
        candidates.append(layer.weight)

    if hasattr(layer, 'modules'):
        for sublayer in layer.modules():
            if sublayer is layer:
                continue
            if hasattr(sublayer, 'weight'):
                candidates.append(sublayer.weight)

    for w in candidates:
        if hasattr(w, 'detach'):
            shape = tuple(w.detach().cpu().shape)
        else:
            shape = tuple(getattr(w, 'shape', ()))
        if len(shape) == 2:
            return shape

    return None


def layer_min_matrix_dim(layer):
    shape = _extract_weight_shape(layer)
    if shape is None:
        return 0
    return int(min(shape))


def ww_alpha_for_matrix(layer, matrix_name):
    watcher = ww.WeightWatcher(model=layer)
    details = watcher.analyze(randomize=True, ERG=True, plot=False)
    if 'alpha' not in details.columns:
        raise RuntimeError(f"WeightWatcher output missing alpha for {matrix_name}: columns={list(details.columns)}")
    return float(details['alpha'].iloc[0])


def convert_matrix_layer(model, Xtr, ytr, matrix_name, train_params=None, num_boost_round=None):
    return convert(
        model,
        Xtr,
        ytr,
        W=matrix_name,
        return_type='torch',
        train_params=train_params,
        num_boost_round=num_boost_round,
    )


def is_small_matrix_error(err):
    msg = str(err)
    return ('minimum of 2 is required by TruncatedSVD' in msg) or ('Found array with 1 feature(s)' in msg)


def evaluate_model(booster, Xtr, ytr, Xte, yte):
    dtr = xgb.DMatrix(Xtr, label=ytr)
    dte = xgb.DMatrix(Xte, label=yte)

    m_tr = booster.predict(dtr, output_margin=True).astype(np.float32)
    p_tr = 1.0 / (1.0 + np.exp(-m_tr))
    tr_acc = float(accuracy_score(ytr, (p_tr >= 0.5).astype(int)))

    m_te = booster.predict(dte, output_margin=True).astype(np.float32)
    p_te = 1.0 / (1.0 + np.exp(-m_te))
    te_acc = float(accuracy_score(yte, (p_te >= 0.5).astype(int)))
    te_loss = float(log_loss(yte, np.vstack([1 - p_te, p_te]).T, labels=[0, 1]))
    return tr_acc, te_acc, te_loss


def save_checkpoint(state, metrics_df, booster):
    STATE_PATH.write_text(json.dumps(state, indent=2))
    metrics_df.to_csv(METRICS_PATH, index=False)
    booster.save_model(str(MODEL_PATH))


## 4) Load OpenML dataset 1049, prepare train/test split, and resume if needed


In [ ]:
filters = Filters(min_rows=100, max_rows=None, max_features=100000)
X, y, meta = load_dataset('openml:1049', filters=filters)

y = np.asarray(y)
if len(np.unique(y)) != 2:
    raise ValueError(f'Dataset is not binary classification; classes={np.unique(y)}')

if hasattr(X, 'tocsr'):
    X = X.tocsr().astype(np.float32)
else:
    X = np.asarray(X, dtype=np.float32)

if FORCE_FRESH_START:
    for p in [STATE_PATH, METRICS_PATH, MODEL_PATH, SPLIT_PATH, SUMMARY_PATH]:
        if p.exists():
            p.unlink()

if SPLIT_PATH.exists():
    npz = np.load(SPLIT_PATH, allow_pickle=False)
    tr_idx, te_idx = npz['tr_idx'], npz['te_idx']
else:
    all_idx = np.arange(len(y))
    tr_idx, te_idx = train_test_split(
        all_idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    np.savez_compressed(SPLIT_PATH, tr_idx=tr_idx, te_idx=te_idx)

Xtr, Xte = X[tr_idx], X[te_idx]
ytr, yte = y[tr_idx], y[te_idx]

if isinstance(meta, dict):
    dataset_name = meta.get('dataset_name', meta.get('name', 'unknown'))
    dataset_uid = meta.get('dataset_uid', meta.get('uid', 'unknown'))
else:
    dataset_name = getattr(meta, 'dataset_name', getattr(meta, 'name', 'unknown'))
    dataset_uid = getattr(meta, 'dataset_uid', getattr(meta, 'uid', 'unknown'))

print('dataset_name:', dataset_name)
print('dataset_uid:', dataset_uid)
print('X shape:', X.shape, '| train:', Xtr.shape, '| test:', Xte.shape)


## 5) Training loop (checkpoint/resume each round) until alpha(W7) and alpha(W8) <= 2


In [ ]:
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 1.0,
    'seed': RANDOM_STATE,
}

state = {
    'dataset_uid': 'openml:1049',
    'target_alpha': TARGET_ALPHA,
    'max_rounds': MAX_ROUNDS,
    'checkpoint_every_rounds': CHECKPOINT_EVERY_ROUNDS,
    'last_completed_round': 0,
    'target_round_w7': None,
    'target_round_w8': None,
    'target_round_both': None,
}

if STATE_PATH.exists() and METRICS_PATH.exists() and MODEL_PATH.exists():
    state = json.loads(STATE_PATH.read_text())
    df_metrics = pd.read_csv(METRICS_PATH)
    booster = xgb.Booster()
    booster.load_model(str(MODEL_PATH))
    start_round = int(state.get('last_completed_round', 0))
    print(f"Resuming from round {start_round}")
else:
    df_metrics = pd.DataFrame()
    booster = None
    start_round = 0
    print('Starting from scratch')

dtr = xgb.DMatrix(Xtr, label=ytr)

MIN_WW_EIGENVALUES = 10  # Wait until constructed matrix has enough spectral support

start_time = time.time()
for r in range(start_round + 1, MAX_ROUNDS + 1):
    booster = xgb.train(
        params=params,
        dtrain=dtr,
        num_boost_round=1,
        xgb_model=booster,
        verbose_eval=False,
    )

    if r % CHECKPOINT_EVERY_ROUNDS != 0:
        continue

    tr_acc, te_acc, te_loss = evaluate_model(booster, Xtr, ytr, Xte, yte)

    try:
        layer_w7 = convert_matrix_layer(booster, Xtr, ytr, 'W7', train_params=params, num_boost_round=r)
        layer_w8 = convert_matrix_layer(booster, Xtr, ytr, 'W8', train_params=params, num_boost_round=r)
    except ValueError as err:
        if is_small_matrix_error(err):
            state['last_completed_round'] = r
            save_checkpoint(state, df_metrics, booster)
            print(f"round={r:4d} | skipping WW: matrix too small for SVD ({err})")
            continue
        raise

    min_dim_w7 = layer_min_matrix_dim(layer_w7)
    min_dim_w8 = layer_min_matrix_dim(layer_w8)

    if min(min_dim_w7, min_dim_w8) == 0:
        state['last_completed_round'] = r
        save_checkpoint(state, df_metrics, booster)
        print(
            f"round={r:4d} | skipping WW: no 2D weight matrix found "
            f"(types: W7={type(layer_w7).__name__}, W8={type(layer_w8).__name__})"
        )
        continue

    if min(min_dim_w7, min_dim_w8) < MIN_WW_EIGENVALUES:
        state['last_completed_round'] = r
        save_checkpoint(state, df_metrics, booster)
        print(
            f"round={r:4d} | warming up "
            f"(min_dim W7={min_dim_w7}, W8={min_dim_w8}; need >= {MIN_WW_EIGENVALUES})"
        )
        continue

    alpha_w7 = ww_alpha_for_matrix(layer_w7, 'W7')
    alpha_w8 = ww_alpha_for_matrix(layer_w8, 'W8')

    rec = {
        'round': r,
        'alpha_w7': alpha_w7,
        'alpha_w8': alpha_w8,
        'train_acc': tr_acc,
        'test_acc': te_acc,
        'test_logloss': te_loss,
        'elapsed_min': (time.time() - start_time) / 60.0,
    }
    df_metrics = pd.concat([df_metrics, pd.DataFrame([rec])], ignore_index=True)

    if state.get('target_round_w7') is None and np.isfinite(alpha_w7) and alpha_w7 <= TARGET_ALPHA:
        state['target_round_w7'] = r
    if state.get('target_round_w8') is None and np.isfinite(alpha_w8) and alpha_w8 <= TARGET_ALPHA:
        state['target_round_w8'] = r
    if state.get('target_round_both') is None and np.isfinite(alpha_w7) and np.isfinite(alpha_w8):
        if alpha_w7 <= TARGET_ALPHA and alpha_w8 <= TARGET_ALPHA:
            state['target_round_both'] = r

    state['last_completed_round'] = r
    save_checkpoint(state, df_metrics, booster)

    print(
        f"round={r:4d} | alpha(W7)={alpha_w7:.3f} alpha(W8)={alpha_w8:.3f} "
        f"| test_acc={te_acc:.4f} logloss={te_loss:.4f}"
    )

    if state.get('target_round_both') is not None:
        print(f"Reached alpha <= {TARGET_ALPHA} for both W7 and W8 at round {r}.")
        break

summary = {
    'dataset_uid': 'openml:1049',
    'target_alpha': TARGET_ALPHA,
    'target_round_w7': state.get('target_round_w7'),
    'target_round_w8': state.get('target_round_w8'),
    'target_round_both': state.get('target_round_both'),
    'final_round': int(state.get('last_completed_round', 0)),
    'checkpoint_dir': str(DRIVE_ROOT),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print('\nSummary:', json.dumps(summary, indent=2))


## 6) Review saved metrics and milestone rounds


In [ ]:
if METRICS_PATH.exists():
    df = pd.read_csv(METRICS_PATH)
    display(df.tail(10))

    print('Rows saved:', len(df))
    if len(df):
        print('Best (min) alpha W7:', float(df['alpha_w7'].min()))
        print('Best (min) alpha W8:', float(df['alpha_w8'].min()))

        m7 = df[df['alpha_w7'] <= TARGET_ALPHA]
        m8 = df[df['alpha_w8'] <= TARGET_ALPHA]
        mb = df[(df['alpha_w7'] <= TARGET_ALPHA) & (df['alpha_w8'] <= TARGET_ALPHA)]

        print('First round alpha(W7) <= target:', None if m7.empty else int(m7['round'].iloc[0]))
        print('First round alpha(W8) <= target:', None if m8.empty else int(m8['round'].iloc[0]))
        print('First round BOTH <= target:', None if mb.empty else int(mb['round'].iloc[0]))
else:
    print('No metrics checkpoint yet.')
